In [ ]:
# Cell 1: Mount Google Drive (holds the real datasets at
# /content/drive/MyDrive/plantguard-data/, per src/config.py).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 2: Clone or pull the private repo using a PAT entered via getpass
# (the token is never hardcoded or printed/logged), then purge any
# already-imported src.* modules so a stale cached module from an earlier
# cell run in this session can never silently run instead of the code just
# pulled, and print the commit actually checked out.
import os
import subprocess
import sys
from getpass import getpass

REPO_DIR = "/content/plantguard-v2"
pat = getpass("GitHub Personal Access Token: ")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    remote = f"https://{pat}@github.com/RUDRAIndia/plantguard-v2.git"
    subprocess.run(["git", "clone", remote, REPO_DIR], check=True)

del pat
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

purged = sorted(name for name in sys.modules if name.startswith("src"))
for name in purged:
    del sys.modules[name]
print(f"Purged {len(purged)} cached src module(s): {purged}")

commit_hash = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"Checked out commit {commit_hash}")

In [ ]:
# Cell 3: Kaggle credentials. See src/data/kaggle_auth.py for the accepted
# kaggle.json shapes, where credentials are installed, and the
# authentication proof that runs before any download starts.
from src.data import kaggle_auth

kaggle_auth.install_credentials()

In [ ]:
# Cell 4: Download PlantVillage and PlantDoc.
from src.data import download

download.main()

In [ ]:
# Cell 5: Build the dataset inventory and the PlantDoc-to-PlantVillage
# class mapping.
from src.data import inventory, mapping_report

inventory.main()
mapping_report.main()

In [ ]:
# Cell 6: Print the inventory report.
from src import config

print((config.ARTIFACTS_DIR / "inventory.md").read_text(encoding="utf-8"))